In [1]:
import sys, json, os
sys.path.append("..")
import pandas as pd
from src.tuning import run_search

X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")["_MICHD"]
final = json.load(open("../reports/final_features.json"))["final"]
print(len(final), "features")                      # expect 14

os.makedirs("../reports/tuning", exist_ok=True)
path = "../reports/best_params.json"
results = json.load(open(path)) if os.path.exists(path) else {}   # survives a kernel restart

def tune(name):
    summary, cvr = run_search(name, X_train, y_train, final)
    results[name] = summary
    cvr.drop(columns=["params"]).to_csv(f"../reports/tuning/{name}_cv_results.csv", index=False)
    json.dump(results, open(path, "w"), indent=2)      # saved after every model
    print(f"{name:7s} PR-AUC {summary['baseline_pr_auc']:.4f} -> {summary['tuned_pr_auc']:.4f}   "
          f"ROC-AUC {summary['baseline_roc_auc']:.4f} -> {summary['tuned_roc_auc']:.4f}   "
          f"({summary['seconds']:.0f}s)")
    print("        best:", summary["best_params"])

14 features


In [2]:
tune("logreg")

logreg  PR-AUC 0.3357 -> 0.3357   ROC-AUC 0.8269 -> 0.8269   (23s)
        best: {'C': 0.03}


In [3]:
tune("dtree")

dtree   PR-AUC 0.3195 -> 0.3232   ROC-AUC 0.8204 -> 0.8222   (53s)
        best: {'max_depth': 12, 'min_samples_leaf': 200}


In [4]:
tune("knn")

knn     PR-AUC 0.2785 -> 0.3164   ROC-AUC 0.7915 -> 0.8176   (11s)
        best: {'n_neighbors': 200}


In [5]:
tune("xgb")

xgb     PR-AUC 0.3285 -> 0.3343   ROC-AUC 0.8254 -> 0.8276   (159s)
        best: {'colsample_bytree': 0.6163856702151521, 'learning_rate': 0.02463975755279057, 'max_depth': 4, 'min_child_weight': 20, 'n_estimators': 600, 'reg_lambda': 18.800303865108354, 'subsample': 0.786705157299192}


In [6]:
tune("rf")

rf      PR-AUC 0.3298 -> 0.3304   ROC-AUC 0.8246 -> 0.8251   (284s)
        best: {'n_estimators': 200, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 16}


In [7]:
tune("svm")

svm     PR-AUC 0.2499 -> 0.2514   ROC-AUC 0.6512 -> 0.6535   (307s)
        best: {'C': 1, 'gamma': 0.1}


In [13]:
cols = ["rows", "n_configs", "baseline_pr_auc", "tuned_pr_auc", "tuned_pr_auc_std",
        "baseline_roc_auc", "tuned_roc_auc", "seconds"]
summ = pd.DataFrame(results).T[cols].apply(pd.to_numeric)
summ["pr_auc_gain"] = summ["tuned_pr_auc"] - summ["baseline_pr_auc"]
summ.round(4)

,rows,n_configs,baseline_pr_auc,tuned_pr_auc,tuned_pr_auc_std,baseline_roc_auc,tuned_roc_auc,seconds,pr_auc_gain
logreg,361971,6,0.3357,0.3357,0.0032,0.8269,0.8269,22.9,0.0000
dtree,361971,9,0.3195,0.3232,0.0025,0.8204,0.8222,29.3,0.0036
knn,30000,4,0.2785,0.3164,0.0135,0.7915,0.8176,22.0,0.0379
xgb,100000,15,0.3285,0.3339,0.0046,0.8254,0.8271,259.0,0.0054
rf,100000,12,0.3298,0.3304,0.0034,0.8246,0.8251,284.2,0.0006
svm,30000,9,0.2499,0.2514,0.0103,0.6512,0.6535,306.8,0.0015


In [9]:
pd.read_csv("../reports/tuning/logreg_cv_results.csv")[
    ["param_model__C", "mean_test_pr_auc", "mean_test_roc_auc"]].round(5) 

,param_model__C,mean_test_pr_auc,mean_test_roc_auc
0,0.01,0.33564,0.82692
1,0.03,0.33574,0.82695
2,0.10,0.33573,0.82691
3,0.30,0.33572,0.82692
4,1.00,0.33572,0.82691
5,3.00,0.33572,0.82691


In [10]:
svm = pd.read_csv("../reports/tuning/svm_cv_results.csv")
print(svm.pivot(index="param_model__C", columns="param_model__gamma",
                values="mean_test_roc_auc").round(3))

param_model__gamma  0.001  0.010  0.100
param_model__C                         
0.1                 0.554  0.605  0.651
1.0                 0.529  0.653  0.654
10.0                0.609  0.652  0.666


In [11]:
from src.tuning import SEARCHES

SEARCHES["knn"] = ("grid", {"model__n_neighbors": [200, 400, 800, 1200]}, 30000, 1)
tune("knn")

SEARCHES["dtree"] = ("grid", {"model__max_depth": [12, 16, 20],
                              "model__min_samples_leaf": [200, 400, 800]}, None, -1)
tune("dtree")

knn     PR-AUC 0.2785 -> 0.3164   ROC-AUC 0.7915 -> 0.8176   (22s)
        best: {'n_neighbors': 200}
dtree   PR-AUC 0.3195 -> 0.3232   ROC-AUC 0.8204 -> 0.8222   (29s)
        best: {'max_depth': 12, 'min_samples_leaf': 200}


In [12]:
from scipy.stats import loguniform, uniform
from src.tuning import N_ITER

SEARCHES["xgb"] = ("random", {"model__n_estimators": [600, 1000, 1500],
                              "model__learning_rate": loguniform(0.01, 0.06),
                              "model__max_depth": [3, 4, 5],
                              "model__min_child_weight": [20, 50, 100],
                              "model__subsample": uniform(0.6, 0.4),
                              "model__colsample_bytree": uniform(0.5, 0.4),
                              "model__reg_lambda": loguniform(10, 100)}, 100000, 1)
N_ITER["xgb"] = 15
tune("xgb")

xgb     PR-AUC 0.3285 -> 0.3339   ROC-AUC 0.8254 -> 0.8271   (259s)
        best: {'colsample_bytree': 0.6306163075223341, 'learning_rate': 0.027790201257148026, 'max_depth': 3, 'min_child_weight': 50, 'n_estimators': 600, 'reg_lambda': 24.472440973990114, 'subsample': 0.7085396127095583}


## Tuning findings

Hyperparameters were tuned with 5-fold CV on PR-AUC (logistic regression, decision tree: full training set; XGBoost, Random Forest: 100k-row subsample; KNN, SVM: 30k-row subsample). Each tuned model is compared with its untuned baseline on identical rows and folds. Because rows differ by model, these numbers are not for cross-model ranking; final comparison uses full-data refits.

**Little to gain from tuning.** Logistic regression was insensitive to regularisation (PR-AUC varied by < 0.0002 across C from 0.01 to 3). Random Forest gained 0.0006 and XGBoost 0.005, both within one fold standard deviation; a second, wider XGBoost search found nothing better (0.3339 vs 0.3343), consistent with the performance plateau seen in baseline results.

**Where tuning helped.** KNN improved from 0.279 to 0.316 PR-AUC by using much heavier smoothing (k = 200 vs 25), a gain of about 2.8 standard deviations; the decision tree gained 0.004 (about 1.4 standard deviations) with a deeper tree and larger leaves.

**SVM.** Across a 3×3 grid of C and gamma, the RBF SVM reached ROC-AUC only 0.53–0.67, far below logistic regression (0.83) on the same features; kernel parameters were not the limiting factor. Class imbalance is tested as a cause in the next step.